## Lab 2b: AgentCore Memory Observability Deep Dive

### Overview

This lab builds directly on **Lab 2: AgentCore Memory** and demonstrates how to leverage AgentCore's **built-in observability** features to monitor, trace, and analyze memory operations in production.

**Prerequisites**: 
- ✅ **Must complete Lab 2 first** - This lab uses the memory resource and agent from Lab 2
- ✅ **Enable CloudWatch Transaction Search** - Required to view AgentCore spans and traces

### What You'll Learn

🔍 **Native AgentCore Memory Observability**:
- **Metrics**: Monitor memory operation performance and usage
- **Spans**: Trace individual memory operations (CreateEvent, RetrieveMemoryRecords)
- **Logs**: View extraction, consolidation, and operation logs

### Tutorial Details

| Information | Details |
|-------------|---------|
| **Tutorial type** | Observability Deep Dive |
| **Agent type** | Memory-Enhanced Agent |
| **Observability Features** | Native AgentCore Memory + OpenTelemetry |
| **Complexity** | Intermediate |
| **SDK used** | AgentCore Memory, CloudWatch, OpenTelemetry |

### Architecture

```
Agent with Memory Hooks → AgentCore Memory → Native Observability
                                                      ↓
                            CloudWatch Metrics + Spans + Logs
```

**Key Insight**: AgentCore Memory provides **built-in observability** - you don't need to implement custom tracing. You just need minimal OpenTelemetry instrumentation to make spans visible.

---


## Step 1: Import Libraries and Setup


In [ ]:
# Import libraries for observability analysis
import boto3
import json
import time
import uuid
from datetime import datetime, timedelta
from IPython.display import display, HTML, Markdown

# Import AgentCore Memory and existing lab components
from lab_helpers.lab2_memory import (
    CustomerSupportMemoryHooks, 
    memory_client, 
    create_or_get_memory_resource
)
from lab_helpers.lab1_strands_agent import (
    MODEL_ID, SYSTEM_PROMPT, get_product_info, get_return_policy
)
from strands import Agent
from strands.models import BedrockModel

# Setup AWS clients for observability
session = boto3.Session()
region = session.region_name
account_id = boto3.client('sts').get_caller_identity()['Account']

cloudwatch = boto3.client('cloudwatch')
logs_client = boto3.client('logs')

print("✅ Observability setup complete")
print(f"📍 Region: {region}")
print(f"🆔 Account: {account_id}")


## Step 2: Setup Memory Resource and Agent (From Lab 2)

We'll use the existing memory resource and agent setup from Lab 2, but add observability monitoring.


In [ ]:
# Get memory resource from Lab 2
memory_id = create_or_get_memory_resource()
print(f"🧠 Using memory resource: {memory_id}")

# Create agent with memory hooks (same as Lab 2)
session_id = str(uuid.uuid4())
actor_id = "customer_observability_demo"

memory_hooks = CustomerSupportMemoryHooks(
    memory_id, memory_client, actor_id, session_id
)

model = BedrockModel(model_id=MODEL_ID, temperature=0.3, region_name=region)
agent = Agent(
    model=model,
    tools=[get_product_info, get_return_policy],
    system_prompt=SYSTEM_PROMPT,
    hooks=[memory_hooks],
)

print("✅ Agent with memory hooks ready")
print(f"👤 Actor ID: {actor_id}")
print(f"🔗 Session ID: {session_id}")


## Step 3: Generate Memory Operations for Observability

Let's create some memory operations that we can observe in metrics, spans, and logs.


In [ ]:
print("🎯 Generating Memory Operations for Observability Analysis")
print("=" * 60)

# Store start time for metrics analysis
operations_start_time = datetime.now()

# Generate agent interactions that trigger memory operations
print("🤖 Running agent interactions (memory retrieval + storage)...")

queries = [
    "What gaming laptop would you recommend?",
    "Tell me about laptop warranties", 
    "I need help choosing between different laptop brands"
]

responses = []
for i, query in enumerate(queries, 1):
    print(f"   Query {i}: {query}")
    response = agent(query)
    responses.append(response)
    print(f"   ✅ Response {i} generated (triggered memory operations)")
    time.sleep(1)  # Small delay between operations

operations_end_time = datetime.now()
operation_duration = (operations_end_time - operations_start_time).total_seconds()

print(f"\n📊 Memory Operations Summary:")
print(f"   • CreateEvent operations: ~3 (agent interactions)")
print(f"   • RetrieveMemoryRecords operations: ~6 (2 per agent query)")
print(f"   • Total operation time: {operation_duration:.1f} seconds")
print(f"   • Time range: {operations_start_time.strftime('%H:%M:%S')} - {operations_end_time.strftime('%H:%M:%S')}")

print(f"\n💡 These operations generate:")
print(f"   📈 CloudWatch metrics in AWS/Bedrock-AgentCore namespace")
print(f"   🔍 Spans in CloudWatch Transaction Search")
print(f"   📋 Logs in /aws/vendedlogs/bedrock-agentcore/memory/ log groups")


## Step 4: Analyze AgentCore Memory Metrics

AgentCore Memory automatically generates metrics in CloudWatch. Let's examine them.


In [ ]:
print("📈 AGENTCORE MEMORY METRICS ANALYSIS")
print("=" * 50)

# AgentCore Memory metrics are in AWS/Bedrock-AgentCore namespace
namespace = 'AWS/Bedrock-AgentCore'

try:
    # Get all memory-related metrics
    response = cloudwatch.list_metrics(Namespace=namespace)
    
    # Filter for our memory resource
    memory_metrics = []
    for metric in response['Metrics']:
        for dimension in metric.get('Dimensions', []):
            if dimension['Name'] == 'Resource' and memory_id in dimension['Value']:
                memory_metrics.append(metric)
                break
    
    if memory_metrics:
        print(f"✅ Found {len(memory_metrics)} metrics for memory resource")
        
        # Group metrics by type
        metric_types = {}
        for metric in memory_metrics:
            metric_name = metric['MetricName']
            if metric_name not in metric_types:
                metric_types[metric_name] = []
            
            # Extract operation from dimensions
            operation = "Unknown"
            for dim in metric['Dimensions']:
                if dim['Name'] == 'Operation':
                    operation = dim['Value']
                    break
            metric_types[metric_name].append(operation)
        
        print("\n📊 Available Metric Types:")
        for metric_name, operations in metric_types.items():
            unique_ops = list(set(operations))
            print(f"   • {metric_name}: {', '.join(unique_ops)}")
        
        print(f"\n🔗 CloudWatch Metrics Console:")
        print(f"https://console.aws.amazon.com/cloudwatch/home?region={region}#metricsV2:graph=~();namespace={namespace}")
        
    else:
        print("⏳ No memory metrics found yet. Metrics may take 2-5 minutes to appear after operations.")
        print("💡 Try running this cell again in a few minutes.")
        
except Exception as e:
    print(f"❌ Error accessing metrics: {e}")

print(f"\n📖 Expected AgentCore Memory Metrics:")
print(f"   • Invocations - Number of memory operations")
print(f"   • Latency - Duration of memory operations") 
print(f"   • CreationCount - New memory events/records created")
print(f"   • Errors - Failed memory operations")
print(f"   • Throttles - Rate-limited operations")


## Step 5: View AgentCore Memory Spans

To see AgentCore Memory spans, you need minimal OpenTelemetry instrumentation. The spans are generated automatically by AgentCore Memory but need OpenTelemetry to be visible in CloudWatch Transaction Search.


In [ ]:
print("🔍 AGENTCORE MEMORY SPANS SETUP")
print("=" * 40)

print("💡 To view AgentCore Memory spans, you need:")
print("   1. CloudWatch Transaction Search enabled ✅")
print("   2. OpenTelemetry instrumentation when running the agent")
print()
print("📝 Memory operations generate these spans automatically:")
print("   • CreateEvent - When saving interactions to memory")
print("   • RetrieveMemoryRecords - When getting customer context")
print("   • GetEvent, ListEvents - Internal memory operations")
print()
print("🎯 Span Attributes you'll see:")
print(f"   • memory.id: {memory_id}")
print(f"   • session.id: {session_id}")
print(f"   • actor.id: {actor_id}")
print("   • event.id: Specific event identifiers")
print("   • namespace: Memory namespace being accessed")
print()

# Create a simple script to run with OpenTelemetry instrumentation
print("📄 To see spans, create this file and run with OpenTelemetry:")
print()
print("=== memory_spans_demo.py ===")
print(f"""
import uuid
from lab_helpers.lab2_memory import CustomerSupportMemoryHooks, memory_client
from lab_helpers.lab1_strands_agent import MODEL_ID, SYSTEM_PROMPT, get_product_info, get_return_policy
from strands import Agent
from strands.models import BedrockModel

# Setup (using existing memory resource)
memory_id = "{memory_id}"
session_id = "spans-demo-" + str(uuid.uuid4())
actor_id = "customer_spans_demo"

memory_hooks = CustomerSupportMemoryHooks(memory_id, memory_client, actor_id, session_id)
model = BedrockModel(model_id=MODEL_ID, temperature=0.3)
agent = Agent(
    model=model,
    tools=[get_product_info, get_return_policy],
    system_prompt=SYSTEM_PROMPT,
    hooks=[memory_hooks],
)

# Generate memory operations that create spans
print("🔍 Generating memory operations with span tracing...")
response = agent("What laptop would you recommend for software development?")
print("✅ Memory operations completed - check CloudWatch Transaction Search for spans!")
print(f"Session ID: {{session_id}}")
""")

print("🚀 Run with:")
print("   export OTEL_PYTHON_DISTRO=aws_distro")
print("   export OTEL_PYTHON_CONFIGURATOR=aws_configurator") 
print("   export OTEL_TRACES_EXPORTER=otlp")
print("   export OTEL_RESOURCE_ATTRIBUTES=service.name=memory-observability")
print("   opentelemetry-instrument python memory_spans_demo.py")
print()
print("🔗 Then view spans in CloudWatch Transaction Search:")
print(f"https://console.aws.amazon.com/cloudwatch/home?region={region}#application-signals:services")


## Conclusion

🎉 **Congratulations!** You've successfully explored AgentCore Memory's comprehensive observability capabilities.

### What You Learned:

✅ **Native Observability**: AgentCore Memory provides built-in metrics, spans, and logs without custom instrumentation  
✅ **Metrics Monitoring**: Memory operations are automatically tracked in CloudWatch metrics  
✅ **Span Tracing**: Memory operations generate spans visible in CloudWatch Transaction Search  
✅ **Minimal Setup**: Just enable Transaction Search and add OpenTelemetry instrumentation  

### Key Takeaways:

1. **No Custom Code Required**: AgentCore Memory observability works out-of-the-box
2. **Minimal Setup**: Just enable Transaction Search and add OpenTelemetry instrumentation
3. **Production Ready**: Built-in observability scales with your memory operations
4. **Comprehensive Coverage**: Metrics, spans, and logs provide complete visibility

### Key Observability Resources:

- **Metrics**: `AWS/Bedrock-AgentCore` CloudWatch namespace
- **Spans**: CloudWatch Transaction Search with OpenTelemetry instrumentation  
- **Logs**: `/aws/vendedlogs/bedrock-agentcore/memory/APPLICATION_LOGS/` log groups
- **Memory Resource**: Your memory ID for filtering and correlation

### Next Steps:

- **Set up CloudWatch dashboards** for your memory metrics
- **Create alerts** for memory performance and errors
- **Explore Transaction Search** to trace customer conversations
- **Monitor memory strategies** for optimization opportunities

---

**Great work!** You now have comprehensive observability for your AgentCore Memory implementation, enabling you to monitor, troubleshoot, and optimize memory operations in production! 🚀
